# Exam 2 - Big Data: network attack flows with Spark RDDs

Extract the requested information from the dataset below with Spark RDD transformations. "Big Data Fundamentals" course, PUCPR 2021 (individual exam).

### Dataset: network-level attack flows

- File available at /home/dados/ddos/prova.csv
- Flow records of network-level attacks on a computer network (CICFlowMeter features)
- ~1 GB
- ~2 M rows

| #  | Name                       | Description                                                                                      |
|----|----------------------------|--------------------------------------------------------------------------------------------------|
| 0  | Number                     | row number                                                                                       |
| 1  | Flow ID                    | flow identifier                                                                                  |
| 2  | Src IP                     | source IP                                                                                        |
| 3  | Src Port                   | source port                                                                                      |
| 4  | Dst IP                     | destination IP                                                                                   |
| 5  | Dst Port                   | destination port                                                                                 |
| 6  | Protocol                   | protocol                                                                                         |
| 7  | Timestamp                  | timestamp                                                                                        |
| 8  | Flow duration              | flow duration in microseconds                                                                    |
| 9  | total Fwd Packet           | total packets in the forward (client to server) direction                                        |
| 10 | total Bwd packets          | total packets in the backward (server to client) direction                                       |
| 11 | total Length of Fwd Packet | total packet size in the forward direction                                                       |
| 12 | total Length of Bwd Packet | total packet size in the backward direction                                                      |
| 13-20 | Fwd/Bwd Packet Length Min/Max/Mean/Std | packet size statistics per direction                                                 |
| 21 | Flow Bytes/s               | flow bytes per second                                                                            |
| 22 | Flow Packets/s             | flow packets per second                                                                          |
| 23-26 | Flow IAT Mean/Std/Max/Min | inter-arrival time between two packets of the flow                                             |
| 27-31 | Fwd IAT Min/Max/Mean/Std/Total | inter-arrival time in the forward direction                                               |
| 32-36 | Bwd IAT Min/Max/Mean/Std/Total | inter-arrival time in the backward direction                                              |
| 37-40 | Fwd/Bwd PSH/URG Flags    | number of packets with the PSH / URG flag per direction (0 for UDP)                              |
| 41-42 | Fwd/Bwd Header Length    | total bytes used for headers per direction                                                       |
| 43-44 | Fwd/Bwd Packets/s        | packets per second per direction                                                                 |
| 45-49 | Packet Length Min/Max/Mean/Std/Variance | packet length statistics                                                          |
| 50-57 | FIN/SYN/RST/PSH/ACK/URG/CWR/ECE Flag Count | number of packets with each TCP flag                                            |
| 58 | down/Up Ratio              | download/upload ratio                                                                            |
| 59 | Average Packet Size        | mean packet size                                                                                 |
| 60-61 | Fwd/Bwd Segment Size Avg | mean segment size per direction                                                                  |
| 62-67 | Fwd/Bwd Bytes/Bulk, Packet/Bulk, Bulk Rate Avg | bulk transfer statistics per direction                                     |
| 68-71 | Subflow Fwd/Bwd Packets/Bytes | mean packets and bytes per sub-flow and direction                                           |
| 72-73 | Fwd/Bwd Init Win bytes   | bytes sent in the initial window per direction                                                   |
| 74 | Fwd Act Data Pkts          | packets with at least 1 byte of TCP payload in the forward direction                             |
| 75 | Fwd Seg Size Min           | minimum segment size observed in the forward direction                                           |
| 76-79 | Active Min/Mean/Max/Std  | time a flow was active before becoming idle                                                      |
| 80-83 | Idle Min/Mean/Max/Std    | time a flow was idle before becoming active                                                      |
| 84 | Label                      | class of the flow                                                                                |


## Extract the information requested in the exam sheet; each cell states the item it answers

In [1]:
# open the Spark session
import os
os.environ['PYSPARK_PYTHON'] = '/usr/bin/python3'

import pyspark
conf = pyspark.SparkConf()

conf.setMaster('spark://spark-master:7077')

sc = pyspark.SparkContext.getOrCreate()
sc.stop()
sc = pyspark.SparkContext(conf = conf)

In [2]:
# load the file from HDFS into an RDD
rowsRDD = sc.textFile('hdfs://namenode:9000/prova.csv')

### PySpark item 1

In [3]:
# the 5 days with the most connections (from the Timestamp field)
rowsRDD.map(lambda l: l.split(',')[7])\
          .map(lambda l: [(l.split(' ')[0]), 1])\
          .filter(lambda l: l[0] != 'Timestamp')\
          .reduceByKey(lambda x,y: x+y)\
          .sortBy(lambda c: c[1], False)\
          .take(5)

[('20/02/2018', 773676),
 ('16/02/2018', 723196),
 ('22/02/2018', 253894),
 ('21/02/2018', 180242),
 ('03/07/2017', 39505)]

### PySpark item 2

In [4]:
# number of connections lasting between 100 and 200 seconds, per label

In [5]:
def within_limits(x):
    try:
        x = int(x)
        if x < 200 and x > 100:
            return 1
        else: 
            return 0
    except:
        return 0
    
rowsRDD.map(lambda l: [l.split(',')[84], l.split(',')[8]])\
        .map(lambda l: [l[0],within_limits(l[1])])\
        .reduceByKey(lambda x,y: x+y)\
        .filter(lambda l: l[0]!='Label')\
        .collect()

[('ddos', 3986), ('Benign', 22217)]

### PySpark item 3

In [6]:
# number of flows per label
rowsRDD.map(lambda l: [l.split(',')[84], 1])\
    .filter(lambda l: l[0] != 'Label')\
    .reduceByKey(lambda x,y: x+y)\
    .collect()

[('ddos', 1294529), ('Benign', 705470)]

### PySpark item 4

In [7]:
# the 5 most frequent source IPs

In [8]:
rowsRDD.map(lambda l: [l.split(',')[2], 1])\
        .filter(lambda l: l[0] != 'Src IP')\
        .reduceByKey(lambda x,y: x+y)\
        .sortBy(lambda l: l[1], False)\
        .take(5)

[('172.31.69.25', 353151),
 ('18.219.193.20', 348970),
 ('172.31.69.28', 185081),
 ('18.218.229.235', 37035),
 ('18.216.200.189', 36992)]

### PySpark item 5

In [9]:
# longest flow for each label and protocol

In [10]:
# compares durations as strings (values too large to convert safely to int).
def compare(x, y):
    arr = [x,y]
    arr.sort()
    return arr[1]

rowsRDD.map(lambda l: [str(l.split(',')[84])+"-"+str(l.split(',')[6]), l.split(',')[8]])\
       .reduceByKey(lambda x,y: compare(x,y))\
       .collect()

[('Label-Protocol', 'Flow Duration'),
 ('ddos-6', '99999947'),
 ('Benign-0', '99999990'),
 ('Benign-6', '999999'),
 ('Benign-17', '99992'),
 ('ddos-17', '99999830')]

### PySpark item 6

In [11]:
# an IP has 4 bytes in the form BYTE1.BYTE2.BYTE3.BYTE4:
# the 5 most frequent values of BYTE1 in the destination IP.

In [12]:
rowsRDD.map(lambda l: [l.split(',')[4].split('.')[0], 1])\
        .reduceByKey(lambda x,y: x+y)\
        .sortBy(lambda l: l[1], False)\
        .take(5)

[('172', 1182960),
 ('18', 518799),
 ('169', 49418),
 ('52', 32581),
 ('192', 31510)]

### PySpark item 7

In [13]:
# a service is identified by destination IP and destination port:
# the 5 most accessed services.

In [14]:
rowsRDD.map(lambda l: [l.split(',')[4]+"::"+l.split(',')[5], 1])\
        .reduceByKey(lambda x,y: x+y)\
        .sortBy(lambda l: l[1], False)\
        .take(5)

[('172.31.69.25::80', 475265),
 ('172.31.69.28::80', 250274),
 ('172.31.0.2::53', 218991),
 ('169.254.169.254::80', 48885),
 ('172.31.69.25::21', 21245)]

In [15]:
# " ... :: ... " = " IP :: Port"

### PySpark item 8

In [16]:
# number of connections per duration class (Flow Duration)


In [17]:
def duration_class(x):
    try:
        x = int(x)
        
        if x < 101 and x > 0:
            return 'small'
        elif x < 1001:
            return 'medium'
        elif x < 10001:
            return 'large'
        else:
            return 'jumbo'
    except:
        return 'error'
        
rowsRDD.map(lambda l: l.split(',')[8])\
        .map(lambda l: [duration_class(l), 1])\
        .reduceByKey(lambda x,y: x+y)\
        .filter(lambda l: l[0]!='error')\
        .take(5)

[('pequena', 112337),
 ('media', 226615),
 ('jumbo', 1232358),
 ('grande', 428689)]